In [ ]:
# Installing needed package (triton)
! pip install triton

In [ ]:
# Importing needed libraries
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

In [ ]:
# Global Variables
DEVICE = "cuda"
C_OUT = 64
C_IN = 3
H = 1024
W = 1024
FH = 3
FW = 3

In [ ]:
# Making torch tensors
tensor_I = torch.rand(1, C_IN, H, W, device=DEVICE)          # Input  (batch_size=1)
tensor_F = torch.rand(C_OUT, C_IN, FH, FW, device=DEVICE)    # Weights

In [ ]:
# This is the result from Convolutional Layer provided by Torch
# Use this for correctness check
golden_out = F.conv2d(tensor_I, tensor_F, padding=1)
print(golden_out.shape)   # (1, C_OUT, OUT_H, OUT_W)

In [ ]:
# -----------------------------------------------------------------------
# Triton kernel for 2-D convolution
#
# Grid:  (C_OUT, ceil(H / TILE_H), ceil(W / TILE_W))
# Block: TILE_H * TILE_W threads (one per output element in the tile)
#
# Each kernel instance computes a TILE_H x TILE_W patch of one output
# channel.  It loops over all input channels and filter positions,
# loading input and filter elements and accumulating the result.
# -----------------------------------------------------------------------

@triton.jit
def my_triton_kernel(
    # Pointers
    input_ptr,    # (1, C_IN, H, W)         — contiguous
    filter_ptr,   # (C_OUT, C_IN, FH, FW)   — contiguous
    output_ptr,   # (1, C_OUT, H, W)        — contiguous
    # Dimensions
    C_IN:  tl.constexpr,
    C_OUT: tl.constexpr,
    H:     tl.constexpr,
    W:     tl.constexpr,
    FH:    tl.constexpr,
    FW:    tl.constexpr,
    PAD:   tl.constexpr,
    # Tile sizes
    TILE_H: tl.constexpr,
    TILE_W: tl.constexpr,
):
    """
    Triton kernel for Conv2d with padding=1, stride=1.
    Loads values from input and filter, accumulates, stores result.
    """
    # Programme IDs
    oc    = tl.program_id(0)                    # output channel
    tile_h = tl.program_id(1)                   # tile row index
    tile_w = tl.program_id(2)                   # tile col index

    # Output positions covered by this program
    oh_start = tile_h * TILE_H
    ow_start = tile_w * TILE_W

    # Offsets within the tile
    oh_offs = oh_start + tl.arange(0, TILE_H)   # (TILE_H,)
    ow_offs = ow_start + tl.arange(0, TILE_W)   # (TILE_W,)

    # 2-D grid of output coordinates  (TILE_H, TILE_W)
    oh_2d = oh_offs[:, None]    # (TILE_H, 1)
    ow_2d = ow_offs[None, :]    # (1, TILE_W)

    # Validity mask (handles boundary rows/cols)
    mask = (oh_2d < H) & (ow_2d < W)

    # Accumulator
    acc = tl.zeros((TILE_H, TILE_W), dtype=tl.float32)

    for ic in range(C_IN):
        for fr in range(FH):
            for fc in range(FW):
                # Input coordinates (with padding)
                ih = oh_2d + fr - PAD   # (TILE_H, 1)
                iw = ow_2d + fc - PAD   # (1, TILE_W)

                # Boundary check for input
                in_mask = mask & (ih >= 0) & (ih < H) & (iw >= 0) & (iw < W)

                # Load input element
                in_idx = ic * H * W + ih * W + iw   # (TILE_H, TILE_W)
                in_val = tl.load(input_ptr + in_idx, mask=in_mask, other=0.0)

                # Load filter element (scalar)
                flt_idx = oc * C_IN * FH * FW + ic * FH * FW + fr * FW + fc
                flt_val = tl.load(filter_ptr + flt_idx)

                acc += in_val * flt_val

    # Store results
    out_idx = oc * H * W + oh_2d * W + ow_2d   # (TILE_H, TILE_W)
    tl.store(output_ptr + out_idx, acc, mask=mask)


def my_conv2d(input, kernel):
    """
    Wrapper that preprocesses inputs and calls my_triton_kernel.
    input:  torch.Tensor (1, C_IN, H, W)
    kernel: torch.Tensor (C_OUT, C_IN, FH, FW)
    Returns: (output tensor, execution_time_ms)
    """
    # Unpack dimensions
    _, c_in, h, w     = input.shape
    c_out, _, fh, fw  = kernel.shape
    pad               = fh // 2   # padding=1 for 3x3 filter

    # Output tensor (same spatial size due to padding=1)
    output = torch.empty(1, c_out, h, w, device=input.device, dtype=input.dtype)

    # Tile sizes (must be powers-of-2 for Triton)
    TILE_H = 16
    TILE_W = 16

    # Grid: one program per (output_channel, tile_row, tile_col)
    grid = (
        c_out,
        triton.cdiv(h, TILE_H),
        triton.cdiv(w, TILE_W),
    )

    # Measure execution time with CUDA events
    start_event = torch.cuda.Event(enable_timing=True)
    end_event   = torch.cuda.Event(enable_timing=True)

    # Warm-up
    my_triton_kernel[grid](
        input, kernel, output,
        c_in, c_out, h, w, fh, fw, pad,
        TILE_H, TILE_W,
    )
    torch.cuda.synchronize()

    # Timed run
    start_event.record()
    my_triton_kernel[grid](
        input, kernel, output,
        c_in, c_out, h, w, fh, fw, pad,
        TILE_H, TILE_W,
    )
    end_event.record()
    torch.cuda.synchronize()

    execution_time_ms = start_event.elapsed_time(end_event)
    return output, execution_time_ms


In [ ]:
# Testing
# Comparing the result from my_conv2d and Conv from torch
my_output, execution_time = my_conv2d(tensor_I, tensor_F)
torch.testing.assert_close(golden_out, my_output, atol=1e-3, rtol=1e-3)  # Assert should pass
# Printing the execution time
print(f"Execution Time for triton kernel: {execution_time:.4f} ms")